# 04 — Injection Pace Tracker
Tracks current EU injection season vs historical pace and computes the rate needed to hit 90% by November 1.

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c))
        os.chdir(_c)
        print(f"Project root: {_c}")
        break

DATA = Path("data/processed")
df = pd.read_parquet(DATA / "eu_aggregate_full.parquet")
df.index = pd.to_datetime(df.index).tz_localize(None)
df = df.sort_index()
df["year"]  = df.index.year
df["month"] = df.index.month

# Tag injection-season year (Apr 1 - Oct 31): before Apr belongs to prior season
df["inj_season_year"] = df.apply(
    lambda r: r["year"] if r["month"] >= 4 else r["year"] - 1, axis=1
)

print(f"eu_aggregate_full.parquet: {df.shape}")
print(f"  {df.index[0].date()} -> {df.index[-1].date()}")


## 1. Build Season Cumulative Injection Series

In [ ]:
# Build per-season cumulative injection table
season_records = []
for yr in sorted(df["inj_season_year"].unique()):
    season = df[
        (df["inj_season_year"] == yr) &
        (df["month"].isin(range(4, 10)))
    ].copy()
    if season.empty:
        continue
    start = pd.Timestamp(yr, 4, 1)
    season["season_day"] = (season.index - start).days + 1
    season["cum_inj"]    = season["injection"].clip(lower=0).cumsum()
    season["season_year"] = yr
    season_records.append(season)

seasons_df = pd.concat(season_records)
current_season = int(seasons_df["season_year"].max())
historical     = seasons_df[seasons_df["season_year"] < current_season]
this_seas      = seasons_df[seasons_df["season_year"] == current_season].set_index("season_day")

hist_avg = historical.groupby("season_day")["cum_inj"].mean()
hist_p25 = historical.groupby("season_day")["cum_inj"].quantile(0.25)
hist_p75 = historical.groupby("season_day")["cum_inj"].quantile(0.75)

print(f"Current injection season: {current_season}")
print(f"Historical seasons used: {sorted(historical['season_year'].unique())}")
print(f"Current season latest day: {int(this_seas.index.max())} "
      f"({(pd.Timestamp(current_season, 4, 1) + pd.Timedelta(days=int(this_seas.index.max())-1)).date()})")


## 2. Current Season vs Historical Cumulative Injection

In [ ]:
fig = go.Figure()
# Historical P25-P75 band
fig.add_trace(go.Scatter(
    x=list(hist_p75.index) + list(hist_p25.index[::-1]),
    y=list(hist_p75.values) + list(hist_p25.values[::-1]),
    fill="toself", fillcolor="rgba(39,174,96,0.15)",
    line=dict(color="rgba(0,0,0,0)"), name="Historical P25-P75"
))
fig.add_trace(go.Scatter(
    x=hist_avg.index, y=hist_avg.values,
    name="Historical Average", line=dict(color="#27AE60", width=2, dash="dash")
))
fig.add_trace(go.Scatter(
    x=this_seas.index, y=this_seas["cum_inj"],
    name=f"{current_season} Season", line=dict(color="#E74C3C", width=2.5)
))
x_ticks = list(range(1, 215, 30))
x_labels = [
    (pd.Timestamp(2020, 4, 1) + pd.Timedelta(days=d - 1)).strftime("%b %d")
    for d in x_ticks
]
fig.update_layout(
    title=f"Cumulative Injection: {current_season} vs Historical Seasons",
    xaxis=dict(title="Season Day (Apr 1 = Day 1)", tickvals=x_ticks, ticktext=x_labels),
    yaxis_title="Cumulative Injection (GWh)",
    height=460, template="plotly_white"
)
fig.show()


## 3. Days Ahead / Behind Historical Average

In [ ]:
latest_day   = int(this_seas.index.max())
cum_now      = float(this_seas.loc[latest_day, "cum_inj"])
hist_now     = float(hist_avg.loc[latest_day]) if latest_day in hist_avg.index else float("nan")
delta        = cum_now - hist_now
avg_daily    = float(historical.groupby("season_day")["injection"].mean().loc[:latest_day].mean())
days_delta   = delta / avg_daily if avg_daily > 0 else float("nan")
season_date  = (pd.Timestamp(current_season, 4, 1) + pd.Timedelta(days=latest_day - 1)).date()

print(f"Injection pace snapshot — {season_date}")
print(f"{'='*48}")
print(f"  Season day:               {latest_day}")
print(f"  Cumulative injection:      {cum_now:>10,.0f} GWh")
print(f"  Historical avg (same day): {hist_now:>10,.0f} GWh")
print(f"  Delta vs avg:              {delta:>+10,.0f} GWh  ({'ahead' if delta > 0 else 'behind'})")
print(f"  Avg daily rate (hist):     {avg_daily:>10,.0f} GWh/day")
print(f"  Pace gap in days:          {days_delta:>+10.1f} days")


## 4. Required Daily Rate to Reach 90% by Nov 1

In [ ]:
CAPACITY_TWH     = float(df["workingGasVolume"].dropna().iloc[-1])
TARGET_FILL      = 0.90
current_twh      = float(df["gasInStorage"].dropna().iloc[-1])
current_fill_pct = float(df["full"].dropna().iloc[-1])
latest_date      = df["full"].dropna().index[-1]

# Target: Nov 1 of current or next year
target_year  = latest_date.year if latest_date.month < 11 else latest_date.year + 1
TARGET_DATE  = pd.Timestamp(target_year, 11, 1)
days_left    = max((TARGET_DATE - latest_date).days, 1)
needed_gwh   = (CAPACITY_TWH * TARGET_FILL - current_twh) * 1000  # TWh -> GWh
req_rate     = needed_gwh / days_left
recent_rate  = float(df["injection"].dropna().iloc[-30:].mean())

print(f"As of: {latest_date.date()}")
print(f"{'='*48}")
print(f"  Current fill:       {current_fill_pct:.1f}%  ({current_twh:.1f} TWh)")
print(f"  Working capacity:   {CAPACITY_TWH:.1f} TWh")
print(f"  Target (90%) date:  {TARGET_DATE.date()}")
print(f"  Target storage:     {CAPACITY_TWH * TARGET_FILL:.1f} TWh")
print(f"  Still to inject:    {needed_gwh:,.0f} GWh  ({needed_gwh/1000:.1f} TWh)")
print(f"  Days remaining:     {days_left}")
print(f"  Required rate:      {req_rate:,.0f} GWh/day")
print(f"  30d average actual: {recent_rate:,.0f} GWh/day")
print(f"  Rate gap:           {req_rate - recent_rate:+,.0f} GWh/day")


## 5. Pace Scenario Table — Projected Nov 1 Fill

In [ ]:
scenarios = {
    "Current pace":            recent_rate,
    "Current -15%":            recent_rate * 0.85,
    "Current -30%":            recent_rate * 0.70,
    "Required (-> 90%)":       req_rate,
    "Current +15%":            recent_rate * 1.15,
    "Current +30%":            recent_rate * 1.30,
}

rows = []
for label, rate in scenarios.items():
    proj_gwh  = rate * days_left
    proj_twh  = min(current_twh + proj_gwh / 1000, CAPACITY_TWH)
    proj_fill = proj_twh / CAPACITY_TWH * 100
    rows.append({
        "Scenario":              label,
        "Rate (GWh/d)":         f"{rate:>7,.0f}",
        "Proj. Storage (TWh)":  f"{proj_twh:>6.1f}",
        "Proj. Nov 1 Fill":     f"{proj_fill:>5.1f}%",
        "Hits 90%?":            "Yes" if proj_fill >= 90 else " No",
    })

tbl = pd.DataFrame(rows).set_index("Scenario")
print(f"Injection Pace Scenarios — Projected Nov 1 Fill (as of {latest_date.date()})")
print("=" * 65)
print(tbl.to_string())
